In [ ]:
# ==============================================================================
# CELDA 0: RUTAS DEL REPOSITORIO
# Este cuaderno leia sus datos desde Google Drive. Ahora los lee del propio
# repositorio, de modo que corre en cualquier clon sin configuracion previa.
# La raiz se resuelve buscando hacia arriba: funciona igual si el cuaderno se
# ejecuta desde su carpeta o desde la raiz del proyecto.
# ==============================================================================
from pathlib import Path

def _raiz_del_repositorio() -> Path:
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / 'pyproject.toml').exists() and (carpeta / 'quanta').is_dir():
            return carpeta
    raise RuntimeError(
        'No se encontro la raiz del repositorio. Ejecute el cuaderno desde '
        'dentro del proyecto indice-sned.'
    )

RAIZ            = _raiz_del_repositorio()
RUTA_RAW        = (RAIZ / 'data' / 'raw').as_posix() + '/'
RUTA_PROCESADOS = (RAIZ / 'data' / 'processed').as_posix() + '/'
RUTA_REGISTRO   = (RAIZ / 'models' / 'registry').as_posix() + '/'
RUTA_METADATOS  = (RAIZ / 'models' / 'metadata').as_posix() + '/'

print('Raiz del repositorio:', RAIZ)


In [1]:
# ==============================================================================
# CELDA 1: LIBRERÍAS + INGESTA RENDIMIENTO 2018-2019
# ==============================================================================
import pandas as pd

RUTA = RUTA_RAW + 'rendimiento/'

archivos = {
    2018: 'Resumen_Rendimiento 2018.csv',
    2019: 'Resumen_Rendimiento 2019.csv',
}

rend_brutas = {}
for anio, nombre in archivos.items():
    ruta = RUTA + nombre
    for enc in ['utf-8-sig', 'latin-1', 'cp1252']:
        try:
            rend_brutas[anio] = pd.read_csv(ruta, encoding=enc, sep=';', low_memory=False)
            break
        except UnicodeDecodeError:
            continue
    print(f"OK | {anio} | {rend_brutas[anio].shape[0]} filas x {rend_brutas[anio].shape[1]} columnas")

print("\nColumnas 2018:", rend_brutas[2018].columns.tolist())

Mounted at /content/drive
OK | 2018 | 13940 filas x 115 columnas
OK | 2019 | 13885 filas x 124 columnas

Columnas 2018: ['AGNO', 'RBD', 'DGV_RBD', 'NOM_RBD', 'COD_REG_RBD', 'NOM_REG_RBD_A', 'COD_PRO_RBD', 'COD_COM_RBD', 'NOM_COM_RBD', 'COD_DEPROV_RBD', 'NOM_DEPROV_RBD', 'COD_DEPE', 'COD_DEPE2', 'RURAL_RBD', 'ESTADO_ESTAB', 'COD_ENSE', 'COD_ENSE2', 'APR_HOM_01', 'APR_HOM_02', 'APR_HOM_03', 'APR_HOM_04', 'APR_HOM_05', 'APR_HOM_06', 'APR_HOM_07', 'APR_HOM_08', 'APR_HOM_TO', 'APR_MUJ_01', 'APR_MUJ_02', 'APR_MUJ_03', 'APR_MUJ_04', 'APR_MUJ_05', 'APR_MUJ_06', 'APR_MUJ_07', 'APR_MUJ_08', 'APR_MUJ_TO', 'APR_SI_05', 'APR_SI_08', 'APR_SI_TO', 'REP_HOM_01', 'REP_HOM_02', 'REP_HOM_03', 'REP_HOM_04', 'REP_HOM_05', 'REP_HOM_06', 'REP_HOM_07', 'REP_HOM_08', 'REP_HOM_TO', 'REP_MUJ_01', 'REP_MUJ_02', 'REP_MUJ_03', 'REP_MUJ_04', 'REP_MUJ_05', 'REP_MUJ_06', 'REP_MUJ_07', 'REP_MUJ_08', 'REP_MUJ_TO', 'RET_HOM_01', 'RET_HOM_02', 'RET_HOM_03', 'RET_HOM_04', 'RET_HOM_05', 'RET_HOM_06', 'RET_HOM_07', 'RET_HOM_

In [2]:
# ==============================================================================
# CELDA 2: LIMPIEZA Y TASAS DE APROBACIÓN/REPROBACIÓN/RETIRO POR RBD
# ==============================================================================
def procesar_rendimiento(df, anio):
    d = df.copy()
    d['RBD'] = pd.to_numeric(d['RBD'], errors='coerce')
    d = d.dropna(subset=['RBD'])
    d['rbd'] = d['RBD'].astype('Int64').astype(str)

    # Totales por situación (hombre+mujer+sin_info, ya vienen agregados en _TO)
    d['total_aprobados'] = d['APR_HOM_TO'].fillna(0) + d['APR_MUJ_TO'].fillna(0) + d.get('APR_SI_TO', 0)
    d['total_reprobados'] = d['REP_HOM_TO'].fillna(0) + d['REP_MUJ_TO'].fillna(0)
    d['total_retirados'] = d['RET_HOM_TO'].fillna(0) + d['RET_MUJ_TO'].fillna(0)
    d['total_matricula'] = d['total_aprobados'] + d['total_reprobados'] + d['total_retirados']

    d['tasa_aprobacion'] = (d['total_aprobados'] / d['total_matricula']).where(d['total_matricula'] > 0)
    d['tasa_reprobacion'] = (d['total_reprobados'] / d['total_matricula']).where(d['total_matricula'] > 0)
    d['tasa_retiro'] = (d['total_retirados'] / d['total_matricula']).where(d['total_matricula'] > 0)

    return d[['rbd', 'total_matricula', 'tasa_aprobacion', 'tasa_reprobacion', 'tasa_retiro']]

rend_limpios = {anio: procesar_rendimiento(df, anio) for anio, df in rend_brutas.items()}

# Bienio 2018-19: promedio simple por colegio (algunos RBD pueden repetirse por nivel/curso -> agrupar primero)
for anio in rend_limpios:
    rend_limpios[anio] = rend_limpios[anio].groupby('rbd', as_index=False).mean()
    print(f"{anio}: {rend_limpios[anio].shape[0]} colegios únicos")

pool = pd.concat(rend_limpios.values(), ignore_index=True)
rendimiento_1819 = pool.groupby('rbd', as_index=False).mean()

print(f"\nBienio 2018-19: {rendimiento_1819.shape[0]} colegios")
print(rendimiento_1819.describe())

RUTA_SALIDA = RUTA_PROCESADOS
rendimiento_1819.to_parquet(RUTA_SALIDA + 'rendimiento_2018_19_por_rbd.parquet', index=False)
print("Guardado OK")

2018: 9192 colegios únicos
2019: 9114 colegios únicos

Bienio 2018-19: 9219 colegios
       total_matricula  tasa_aprobacion  tasa_reprobacion  tasa_retiro
count      9219.000000      9206.000000       9206.000000  9206.000000
mean        210.626699         0.932353          0.034687     0.032960
std         210.719391         0.112109          0.054074     0.082637
min           0.000000         0.000000          0.000000     0.000000
25%          45.000000         0.928710          0.003535     0.000000
50%         158.500000         0.968754          0.019114     0.006993
75%         300.875000         0.990134          0.044533     0.023427
max        2089.000000         1.000000          1.000000     1.000000
Guardado OK


In [3]:
# ==============================================================================
# CELDA 3: INTEGRAR RENDIMIENTO A LA TABLA DE ENTRENAMIENTO
# ==============================================================================
RUTA = RUTA_PROCESADOS
df_modelo = pd.read_parquet(RUTA + 'tabla_modelo_final_v6.parquet')
rend = pd.read_parquet(RUTA + 'rendimiento_2018_19_por_rbd.parquet')

df_modelo_v7 = pd.merge(df_modelo, rend, on='rbd', how='left', validate='one_to_one')

print(f"Filas: {len(df_modelo_v7)} (antes: {len(df_modelo)})")
print(f"Con dato de rendimiento: {df_modelo_v7['tasa_aprobacion'].notna().sum()}")

df_modelo_v7.to_parquet(RUTA + 'tabla_modelo_final_v7.parquet', index=False)
print(df_modelo_v7.shape)

Filas: 7754 (antes: 7754)
Con dato de rendimiento: 7754
(7754, 55)
